# 06 · CVD · Clinical NLP (`BioBERT`/`ClinicalBERT`), 3-Way Fusion & Consolidated Week 4 Ablations

**Project:** Multimodal AI-based multi-disease clinical decision support system  
**Author:** Member 2 (CVD and CKD)  

**Goals of this notebook:**
1. Fine-tune a transfer-based medical NLP encoder (`emilyalsentzer/Bio_ClinicalBERT` with `dmis-lab/biobert-v1.1` fallback) on `PTB-XL` clinical reports using the exact patient-level `strat_fold` splits from Notebook 01.
2. Calibrate text-model probabilities on validation logits and evaluate **3-way multimodal late fusion** (`ECG + Clinical Tabular + Clinical Report Text`) plus **3-way conflict detection**.
3. Save the fine-tuned text encoder + calibrator bundle (`cvd_text_model/`) and export an upgraded `cvd_assess_multimodal()` function supporting optional free-text reports.
4. Generate the **Consolidated Week 4 Ablation & Evaluation Suite** across both CVD and CKD.

## Methodological Note on PTB-XL Clinical Reports
- `PTB-XL` reports are cardiologist ECG interpretations written primarily in German (~70%) and English (~28%), often containing direct diagnostic phrasing (e.g., *"sinus rhythm"*, *"myocardial infarction"*).
- Because the report is written *after* inspecting the ECG, high standalone text AUROC is expected and reflects diagnostic summarization rather than independent pre-test screening. We explicitly report both **2-way fusion (`ECG + Clinical`)** (pre-report screening) and **3-way fusion (`ECG + Clinical + Report Text`)** (full clinical record synthesis + report-vs-signal consistency checking).

In [1]:
!pip install -q transformers accelerate datasets scikit-learn

import os, re, json, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, accuracy_score, brier_score_loss)

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

DRIVE     = "/content/drive/MyDrive/cdss_data"
PROC_PTB  = f"{DRIVE}/processed/ptbxl"
PROC_UCI  = f"{DRIVE}/processed/uci_heart"
RES_CVD   = f"{DRIVE}/results/cvd"
RES_CKD   = f"{DRIVE}/results/ckd"
RES_ALL   = f"{DRIVE}/results"
MODEL_DIR = f"{DRIVE}/processed/models/cvd"
TEXT_DIR  = f"{MODEL_DIR}/clinical_nlp_encoder"
for d in [RES_CVD, RES_ALL, MODEL_DIR, TEXT_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
assert device == "cuda", "Switch to T4 GPU (Runtime > Change runtime type > T4 GPU) for transformer fine-tuning."

Mounted at /content/drive
Device: cuda


## 1. Load Aligned PTB-XL Subset Metadata & Clinical Reports

We align `ptbxl_subset_meta.csv` row-for-row against `ecg_id` inside `02_test_predictions.npz` so every test report corresponds to the exact same patient and ECG prediction from Notebook 02.

In [2]:
meta_df = pd.read_csv(f"{PROC_PTB}/ptbxl_subset_meta.csv", index_col="ecg_id")
npz_02  = np.load(f"{RES_CVD}/02_test_predictions.npz", allow_pickle=True)
CLASSES = [str(c) for c in npz_02["classes"]]

# Clean text column
text_col = "report_clean" if "report_clean" in meta_df.columns else "report"
meta_df["text"] = meta_df[text_col].fillna("").astype(str).str.strip()
meta_df.loc[meta_df["text"] == "", "text"] = "no clinical report provided"

tr_df = meta_df[meta_df["split"] == "train"].copy()
va_df = meta_df[meta_df["split"] == "val"].copy()

# Align test split strictly to ecg_id order in 02_test_predictions.npz
test_ecg_ids = npz_02["ecg_id"]
te_df = meta_df.loc[test_ecg_ids].copy()
assert np.array_equal(te_df.index.values, test_ecg_ids), "Test ecg_id alignment mismatch!"
assert np.array_equal(te_df[CLASSES].values.astype(int), npz_02["Y_test"].astype(int)), "Label mismatch!"

print(f"Aligned splits -> train: {len(tr_df)} | val: {len(va_df)} | test: {len(te_df)}")
print(f"Non-empty report rate -> train: {(tr_df['text'] != 'no clinical report provided').mean()*100:.1f}% | "
      f"test: {(te_df['text'] != 'no clinical report provided').mean()*100:.1f}%")
print("\nSample clinical reports:")
for idx, r in te_df.head(3).iterrows():
    print(f"  [ecg_id={idx} | labels={[c for c in CLASSES if r[c]==1]}] {r['text'][:110]}")

Aligned splits -> train: 5000 | val: 1000 | test: 1500
Non-empty report rate -> train: 100.0% | test: 99.9%

Sample clinical reports:
  [ecg_id=10984 | labels=['MI', 'STTC']] sinus rhythm. poor progression of r waves in anterior chest leads. this suggests old anteroseptal myocardial i
  [ecg_id=9897 | labels=['STTC']] sinusrytm ospecifik t-fÖrÄndring
  [ecg_id=1343 | labels=['MI']] sinusrhythmus qrs(t) abnormal    inferiorer infarkt     wahrscheinlich alt


## 2. Fine-Tune Medical Transformer (`Bio_ClinicalBERT` / `BioBERT`)

- **Backbone:** `emilyalsentzer/Bio_ClinicalBERT` (initialized from BioBERT and pretrained on MIMIC-III clinical notes; automatic fallback to `dmis-lab/biobert-v1.1`).
- **Task:** 5-class multi-label classification (`NORM`, `MI`, `STTC`, `CD`, `HYP`) with `BCEWithLogitsLoss`, linear warmup, and best validation macro AUROC checkpointing.
- **Post-hoc Calibration:** Validation logits are calibrated per class via Platt scaling (`LogisticRegression`) so text probabilities match the calibrated scale of `ECGNet` and `UCI Heart`.

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BACKBONE_CANDIDATES = ["emilyalsentzer/Bio_ClinicalBERT", "dmis-lab/biobert-v1.1"]
MODEL_NAME = None
for cand in BACKBONE_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(cand)
        MODEL_NAME = cand
        break
    except Exception as e:
        print(f"Could not load {cand}: {e}")
assert MODEL_NAME is not None, "Failed to load medical transformer backbone."
print("Loaded medical NLP backbone:", MODEL_NAME)

MAX_LEN = 96
BATCH_NLP = 32
EPOCHS_NLP = 4
LR_NLP = 3e-5


class ReportDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len=MAX_LEN):
        self.enc = tok(list(texts), truncation=True, padding="max_length",
                       max_length=max_len, return_tensors="pt")
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {
            "input_ids": self.enc["input_ids"][i],
            "attention_mask": self.enc["attention_mask"][i],
            "labels": self.y[i],
        }


tr_ds = ReportDataset(tr_df["text"], tr_df[CLASSES].values, tokenizer)
va_ds = ReportDataset(va_df["text"], va_df[CLASSES].values, tokenizer)
te_ds = ReportDataset(te_df["text"], te_df[CLASSES].values, tokenizer)

tr_loader = DataLoader(tr_ds, batch_size=BATCH_NLP, shuffle=True)
va_loader = DataLoader(va_ds, batch_size=64, shuffle=False)
te_loader = DataLoader(te_ds, batch_size=64, shuffle=False)

nlp_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), problem_type="multi_label_classification"
).to(device)

optimizer = torch.optim.AdamW(nlp_model.parameters(), lr=LR_NLP, weight_decay=0.01)
total_steps = len(tr_loader) * EPOCHS_NLP
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                            num_training_steps=total_steps)
loss_fn = nn.BCEWithLogitsLoss()


@torch.no_grad()
def eval_nlp_logits(model, loader):
    model.eval()
    logits_list = []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        out = model(input_ids=ids, attention_mask=mask).logits
        logits_list.append(out.cpu().numpy())
    return np.concatenate(logits_list, axis=0)


def macro_auc(Y_true, P_prob):
    aucs = []
    for j in range(Y_true.shape[1]):
        if 0 < Y_true[:, j].sum() < len(Y_true):
            aucs.append(roc_auc_score(Y_true[:, j], P_prob[:, j]))
    return float(np.mean(aucs))


Y_va = va_df[CLASSES].values.astype(int)
Y_te = te_df[CLASSES].values.astype(int)

best_val_auc = -1.0
best_weights = None

for ep in range(EPOCHS_NLP):
    nlp_model.train()
    running_loss = 0.0
    for batch in tr_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = nlp_model(input_ids=ids, attention_mask=mask).logits
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nlp_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * len(labels)

    Lv = eval_nlp_logits(nlp_model, va_loader)
    Pv = 1.0 / (1.0 + np.exp(-np.clip(Lv, -30, 30)))
    v_auc = macro_auc(Y_va, Pv)
    print(f"Epoch {ep+1}/{EPOCHS_NLP} | Train Loss: {running_loss/len(tr_ds):.4f} | Val Macro AUROC: {v_auc:.4f}")
    if v_auc > best_val_auc:
        best_val_auc = v_auc
        best_weights = copy.deepcopy(nlp_model.state_dict())

nlp_model.load_state_dict(best_weights)

# Calibrate text probabilities on validation logits
Lv_best = eval_nlp_logits(nlp_model, va_loader)
Lt_best = eval_nlp_logits(nlp_model, te_loader)

Pv_text = np.zeros_like(Lv_best, dtype=np.float32)
Pt_text = np.zeros_like(Lt_best, dtype=np.float32)
text_cal_params = []

for j in range(len(CLASSES)):
    lr_cal = LogisticRegression(C=1e6, solver="lbfgs").fit(Lv_best[:, [j]], Y_va[:, j])
    Pv_text[:, j] = lr_cal.predict_proba(Lv_best[:, [j]])[:, 1]
    Pt_text[:, j] = lr_cal.predict_proba(Lt_best[:, [j]])[:, 1]
    text_cal_params.append({"coef": float(lr_cal.coef_[0, 0]), "intercept": float(lr_cal.intercept_[0])})

# Evaluate Text-Only per class and macro
text_rows = []
for j, c in enumerate(CLASSES):
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        f = f1_score(Y_va[:, j], (Pv_text[:, j] >= t).astype(int), zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    pred = (Pt_text[:, j] >= best_t).astype(int)
    text_rows.append({
        "class": c,
        "auroc": roc_auc_score(Y_te[:, j], Pt_text[:, j]),
        "f1": f1_score(Y_te[:, j], pred, zero_division=0),
        "precision": precision_score(Y_te[:, j], pred, zero_division=0),
        "recall": recall_score(Y_te[:, j], pred, zero_division=0),
        "accuracy": accuracy_score(Y_te[:, j], pred),
        "brier": brier_score_loss(Y_te[:, j], Pt_text[:, j]),
    })

text_res_df = pd.DataFrame(text_rows)
print("\nCalibrated Medical NLP (Test Split Per Class):")
display(text_res_df.round(3))
print("Text-Only Macro AUROC:", round(text_res_df["auroc"].mean(), 3),
      "| Macro F1:", round(text_res_df["f1"].mean(), 3),
      "| Macro Brier:", round(text_res_df["brier"].mean(), 4))

text_res_df.to_csv(f"{RES_CVD}/06_nlp_text_results_per_class.csv", index=False)
np.savez(f"{RES_CVD}/06_nlp_test_predictions.npz", ecg_id=test_ecg_ids, Y_test=Y_te,
         P_text=Pt_text, classes=np.array(CLASSES))

# Save fine-tuned model, tokenizer, and calibration metadata
nlp_model.save_pretrained(TEXT_DIR)
tokenizer.save_pretrained(TEXT_DIR)
json.dump({"backbone": MODEL_NAME, "classes": CLASSES, "max_len": MAX_LEN, "cal_params": text_cal_params},
          open(f"{TEXT_DIR}/nlp_meta.json", "w"), indent=2)
print("Saved fine-tuned medical NLP encoder to:", TEXT_DIR)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Loaded medical NLP backbone: emilyalsentzer/Bio_ClinicalBERT


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (4 outputs left out)

,class,auroc,f1,precision,recall,accuracy,brier
0,NORM,0.950,0.870,0.908,0.835,0.889,0.084
1,MI,0.928,0.749,0.692,0.816,0.859,0.087
2,STTC,0.900,0.711,0.719,0.703,0.864,0.101
3,CD,0.931,0.781,0.828,0.739,0.905,0.075
4,HYP,0.916,0.686,0.745,0.635,0.927,0.058


Text-Only Macro AUROC: 0.925 | Macro F1: 0.759 | Macro Brier: 0.081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned medical NLP encoder to: /content/drive/MyDrive/cdss_data/processed/models/cvd/clinical_nlp_encoder


## 3. 3-Way Multimodal Fusion (`ECG + Clinical Tabular + Report Text`) & Conflict Detection

We extend Notebook 04's simulated paired test cohort ($N = 3,000$ drawn strictly from held-out test splits) with the calibrated `Bio_ClinicalBERT` text probability $p_{\text{text}} = 1 - P_{\text{text}}(\text{NORM})$ paired to its true ECG record.
- **Single-Modality Baselines:** `Clinical Tabular only`, `ECG only`, `Report Text only`
- **2-Way Multimodal Fusion:** `ECG + Clinical` (pre-report screening scenario)
- **3-Way Multimodal Fusion:** `ECG + Clinical + Report Text` (full clinical record synthesis)
- **3-Way Conflict Detection:** Flags directional disagreement across any pair of available modalities (`ECG vs Clinical`, `ECG vs Text`, `Clinical vs Text`).

In [4]:
import joblib

NORM_IDX = CLASSES.index("NORM")
P_ecg = npz_02["ecg_meta_seed42"]
y_ecg = 1 - Y_te[:, NORM_IDX]
p_ecg_abn = np.clip(1.0 - P_ecg[:, NORM_IDX], 1e-4, 1.0 - 1e-4)
p_txt_abn = np.clip(1.0 - Pt_text[:, NORM_IDX], 1e-4, 1.0 - 1e-4)

# Load UCI Heart clinical test predictions
t_uci = np.load(f"{RES_CVD}/03_uci_test_predictions.npz", allow_pickle=True)
uci_bundle = joblib.load(f"{MODEL_DIR}/uci_heart_models.joblib")
df_uci = pd.read_csv(f"{PROC_UCI}/uci_heart_clean.csv")
UCI_FEATS = uci_bundle["features"]
cal_uci = uci_bundle["model"]
X_tab = df_uci.iloc[t_uci["idx"]][UCI_FEATS].reset_index(drop=True)
y_tab = t_uci["y"].astype(int)

rng = np.random.default_rng(SEED)
N_SIM = 3000
s = rng.integers(0, 2, N_SIM)

def draw_paired(y):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    return np.where(s == 1, rng.choice(pos, N_SIM), rng.choice(neg, N_SIM))

ei, ti = draw_paired(y_ecg), draw_paired(y_tab)
pe = p_ecg_abn[ei]
px = p_txt_abn[ei]  # Same PTB-XL patient report paired with their ECG

X_full = X_tab.iloc[ti].reset_index(drop=True).astype(float)
miss_frac = rng.uniform(0, 0.5, N_SIM)
X_miss = X_full.mask(rng.random(X_full.shape) < miss_frac[:, None])
n_missing = X_miss.isna().sum(axis=1).values

pt_full = np.clip(cal_uci.predict_proba(X_full)[:, 1], 1e-4, 1.0 - 1e-4)
pt_miss = np.clip(cal_uci.predict_proba(X_miss)[:, 1], 1e-4, 1.0 - 1e-4)


def logit(p): return np.log(p / (1.0 - p))
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


def auc_ci(y, p, reps=200):
    r = np.random.default_rng(0); a = []
    for _ in range(reps):
        i = r.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(y):
            a.append(roc_auc_score(y[i], p[i]))
    return roc_auc_score(y, p), np.percentile(a, 2.5), np.percentile(a, 97.5)


# Compare Single -> 2-Way -> 3-Way Fusion (under realistic 0-50% missing clinical inputs)
candidates = [
    ("Clinical Tabular only (0-50% missing)", pt_miss),
    ("ECG Signal only (1D ResNet)", pe),
    ("Clinical Report Text only (Bio_ClinicalBERT)", px),
    ("2-Way Fusion: ECG + Clinical (Log-Odds)", sigmoid(logit(pe) + logit(pt_miss))),
    ("3-Way Fusion: ECG + Clinical + Text (Average)", (pe + pt_miss + px) / 3.0),
    ("3-Way Fusion: ECG + Clinical + Text (Log-Odds)", sigmoid(logit(pe) + logit(pt_miss) + logit(px))),
]

fusion_3way_rows = []
for name, p in candidates:
    a, lo, hi = auc_ci(s, p)
    pred = (p >= 0.5).astype(int)
    fusion_3way_rows.append({
        "setting": name,
        "AUROC (95% CI)": f"{a:.3f} ({lo:.3f} to {hi:.3f})",
        "auroc_num": round(a, 4),
        "accuracy": round(accuracy_score(s, pred), 3),
        "f1": round(f1_score(s, pred), 3),
        "brier": round(brier_score_loss(s, p), 4),
    })

fusion_3way_df = pd.DataFrame(fusion_3way_rows).set_index("setting")
print("3-Way Multimodal Progression (Test Cohort with Realistic Missing Clinical Labs):")
display(fusion_3way_df.drop(columns=["auroc_num"]))
fusion_3way_df.to_csv(f"{RES_CVD}/06_cvd_3way_fusion_results.csv")

# 3-Way Pairwise Conflict Detection
def any_conflict_3way(pe_, pt_, px_, margin=0.1):
    def pair_disagree(a, b):
        return ((a >= 0.5 + margin) & (b <= 0.5 - margin)) | ((b >= 0.5 + margin) & (a <= 0.5 - margin))
    return pair_disagree(pe_, pt_) | pair_disagree(pe_, px_) | pair_disagree(pt_, px_)

pf3 = sigmoid(logit(pe) + logit(pt_miss) + logit(px))
cf3 = any_conflict_3way(pe, pt_miss, px)
c_rows = []
for lbl, mask in [("Agree / No Cross-Modal Conflict", ~cf3), ("CONFLICT (Any Pair Disagrees)", cf3)]:
    c_rows.append({
        "group": lbl,
        "n": int(mask.sum()),
        "share": f"{mask.mean()*100:.1f}%",
        "3-way fused accuracy": round(accuracy_score(s[mask], (pf3[mask] >= 0.5).astype(int)), 3),
        "ECG accuracy": round(accuracy_score(s[mask], (pe[mask] >= 0.5).astype(int)), 3),
        "Clinical accuracy": round(accuracy_score(s[mask], (pt_miss[mask] >= 0.5).astype(int)), 3),
        "Text accuracy": round(accuracy_score(s[mask], (px[mask] >= 0.5).astype(int)), 3),
    })
conf3_df = pd.DataFrame(c_rows).set_index("group")
print("\n3-Way Cross-Modal Conflict Stratification:")
display(conf3_df)
conf3_df.to_csv(f"{RES_CVD}/06_cvd_3way_conflict_detection.csv")

3-Way Multimodal Progression (Test Cohort with Realistic Missing Clinical Labs):


,AUROC (95% CI),accuracy,f1,brier
setting,,,,
Clinical Tabular only (0-50% missing),0.827 (0.810 to 0.842),0.723,0.763,0.1896
ECG Signal only (1D ResNet),0.941 (0.933 to 0.948),0.866,0.863,0.0962
Clinical Report Text only (Bio_ClinicalBERT),0.949 (0.940 to 0.957),0.876,0.884,0.0878
2-Way Fusion: ECG + Clinical (Log-Odds),0.967 (0.962 to 0.972),0.879,0.886,0.0861
3-Way Fusion: ECG + Clinical + Text (Average),0.983 (0.979 to 0.987),0.929,0.932,0.0814
3-Way Fusion: ECG + Clinical + Text (Log-Odds),0.985 (0.981 to 0.988),0.928,0.931,0.0549



3-Way Cross-Modal Conflict Stratification:


,n,share,3-way fused accuracy,ECG accuracy,Clinical accuracy,Text accuracy
group,,,,,,
Agree / No Cross-Modal Conflict,2052,68.4%,0.962,0.938,0.883,0.948
CONFLICT (Any Pair Disagrees),948,31.6%,0.855,0.709,0.376,0.722


## 4. Consolidated Week 4 Ablation & Evaluation Table (CVD & CKD)

Compiles all experimental stages required by the Synopsis (Section 11) and Week 4 Plan (`Single Modality` $\rightarrow$ `Multimodal` $\rightarrow$ `Calibrated/Optimized Multimodal` $\rightarrow$ `+ Reliability & Conflict Filtering` $\rightarrow$ `Cross-Dataset Generalization`) into one unified table for the final report and Research Results dashboard page.

In [5]:
# Load previously verified CVD and CKD CSVs
cvd_02 = pd.read_csv(f"{RES_CVD}/02_baseline_summary_macro.csv")
cvd_03 = pd.read_csv(f"{RES_CVD}/03_uci_test_results.csv")
cvd_04_rel = pd.read_csv(f"{RES_CVD}/04_reliability_levels.csv")
ckd_05_test = pd.read_csv(f"{RES_CKD}/05_ckd_336_test_results.csv")
ckd_05_cross = pd.read_csv(f"{RES_CKD}/05_ckd_cross_dataset_generalization.csv")

ablation_records = [
    # --- CVD Progression ---
    {"Disease": "CVD", "Stage": "1. Single Modality (Demographics only)", "Dataset / Split": "PTB-XL Test (n=1,500)",
     "Model": "Logistic Regression (age, sex)", "AUROC": "0.684", "Accuracy": "0.617", "F1": "0.455", "Brier": "0.229"},
    {"Disease": "CVD", "Stage": "1. Single Modality (Clinical Tabular)", "Dataset / Split": "UCI Heart Test (n=184)",
     "Model": "Calibrated Logistic Regression", "AUROC": "0.903", "Accuracy": "0.837", "F1": "0.856", "Brier": "0.120"},
    {"Disease": "CVD", "Stage": "1. Single Modality (12-Lead ECG)", "Dataset / Split": "PTB-XL Test (3-seed)",
     "Model": "1D Residual CNN (ECGNet)", "AUROC": "0.912", "Accuracy": "0.869", "F1": "0.729", "Brier": "0.090"},
    {"Disease": "CVD", "Stage": "1. Single Modality (Clinical Report Text)", "Dataset / Split": "PTB-XL Test (n=1,500)",
     "Model": f"Calibrated {MODEL_NAME.split('/')[-1]}",
     "AUROC": f"{text_res_df['auroc'].mean():.3f}", "Accuracy": f"{text_res_df['accuracy'].mean():.3f}",
     "F1": f"{text_res_df['f1'].mean():.3f}", "Brier": f"{text_res_df['brier'].mean():.3f}"},
    {"Disease": "CVD", "Stage": "2. Early Multimodal (ECG + Metadata)", "Dataset / Split": "PTB-XL Test (3-seed)",
     "Model": "ECGNet + Age/Sex Fusion", "AUROC": "0.913", "Accuracy": "0.870", "F1": "0.724", "Brier": "0.090"},
    {"Disease": "CVD", "Stage": "3. 2-Way Late Fusion (ECG + Clinical, Missing Labs)", "Dataset / Split": "Simulated Cohort (n=3,000)",
     "Model": "Calibrated Log-Odds Fusion", "AUROC": "0.967", "Accuracy": "0.879", "F1": "0.886", "Brier": "—"},
    {"Disease": "CVD", "Stage": "4. 3-Way Late Fusion (ECG + Clinical + Text)", "Dataset / Split": "Simulated Cohort (n=3,000)",
     "Model": "Calibrated 3-Modality Log-Odds",
     "AUROC": f"{fusion_3way_df.loc['3-Way Fusion: ECG + Clinical + Text (Log-Odds)', 'auroc_num']:.3f}",
     "Accuracy": f"{fusion_3way_df.loc['3-Way Fusion: ECG + Clinical + Text (Log-Odds)', 'accuracy']:.3f}",
     "F1": f"{fusion_3way_df.loc['3-Way Fusion: ECG + Clinical + Text (Log-Odds)', 'f1']:.3f}",
     "Brier": f"{fusion_3way_df.loc['3-Way Fusion: ECG + Clinical + Text (Log-Odds)', 'brier']:.3f}"},
    {"Disease": "CVD", "Stage": "5. + Reliability Filtering (High-Confidence Subset)", "Dataset / Split": "High-Reliability Subset (n=1,747)",
     "Model": "2-Way Fusion + Reliability Gate (>=0.7)", "AUROC": "—", "Accuracy": "0.969", "F1": "—", "Brier": "—"},
    {"Disease": "CVD", "Stage": "6. Cross-Hospital Generalization (LOHO)", "Dataset / Split": "UCI Heart (4 Hospitals Mean)",
     "Model": "Logistic Regression (LOHO)", "AUROC": "0.809", "Accuracy": "—", "F1": "—", "Brier": "—"},

    # --- CKD Progression ---
    {"Disease": "CKD", "Stage": "1. Single Modality (Clinical Labs + History)", "Dataset / Split": "UCI 336 Test (n=80)",
     "Model": "Calibrated Random Forest", "AUROC": "1.000", "Accuracy": "1.000", "F1": "1.000", "Brier": "0.010"},
    {"Disease": "CKD", "Stage": "2. Missing-Data Stress Test (50% Inputs Hidden)", "Dataset / Split": "UCI 336 Test (10 repeats)",
     "Model": "Random Forest + Missing Indicators", "AUROC": "0.982", "Accuracy": "—", "F1": "—", "Brier": "—"},
    {"Disease": "CKD", "Stage": "3. Cross-Dataset Generalization (UCI 336 -> 857)", "Dataset / Split": "22 Overlapping Clinical Features",
     "Model": "XGBoost (Zero-Shot Transfer)", "AUROC": "0.999", "Accuracy": "0.980", "F1": "0.985", "Brier": "0.018"},
    {"Disease": "CKD", "Stage": "4. Cross-Dataset Generalization (UCI 857 -> 336)", "Dataset / Split": "22 Overlapping Clinical Features",
     "Model": "Random Forest (Zero-Shot Transfer)", "AUROC": "1.000", "Accuracy": "0.990", "F1": "0.992", "Brier": "0.016"},
]

consolidated_df = pd.DataFrame(ablation_records)
display(consolidated_df)
consolidated_df.to_csv(f"{RES_ALL}/member2_week4_ablation_summary.csv", index=False)
print("Saved consolidated Week 4 ablation table to:", f"{RES_ALL}/member2_week4_ablation_summary.csv")

,Disease,Stage,Dataset / Split,Model,AUROC,Accuracy,F1,Brier
0,CVD,1. Single Modality (Demographics only),"PTB-XL Test (n=1,500)","Logistic Regression (age, sex)",0.684,0.617,0.455,0.229
1,CVD,1. Single Modality (Clinical Tabular),UCI Heart Test (n=184),Calibrated Logistic Regression,0.903,0.837,0.856,0.120
2,CVD,1. Single Modality (12-Lead ECG),PTB-XL Test (3-seed),1D Residual CNN (ECGNet),0.912,0.869,0.729,0.090
3,CVD,1. Single Modality (Clinical Report Text),"PTB-XL Test (n=1,500)",Calibrated Bio_ClinicalBERT,0.925,0.889,0.759,0.081
4,CVD,2. Early Multimodal (ECG + Metadata),PTB-XL Test (3-seed),ECGNet + Age/Sex Fusion,0.913,0.870,0.724,0.090
5,CVD,"3. 2-Way Late Fusion (ECG + Clinical, Missing ...","Simulated Cohort (n=3,000)",Calibrated Log-Odds Fusion,0.967,0.879,0.886,—
6,CVD,4. 3-Way Late Fusion (ECG + Clinical + Text),"Simulated Cohort (n=3,000)",Calibrated 3-Modality Log-Odds,0.985,0.928,0.931,0.055
7,CVD,5. + Reliability Filtering (High-Confidence Su...,"High-Reliability Subset (n=1,747)",2-Way Fusion + Reliability Gate (>=0.7),—,0.969,—,—
8,CVD,6. Cross-Hospital Generalization (LOHO),UCI Heart (4 Hospitals Mean),Logistic Regression (LOHO),0.809,—,—,—
9,CKD,1. Single Modality (Clinical Labs + History),UCI 336 Test (n=80),Calibrated Random Forest,1.000,1.000,1.000,0.010


Saved consolidated Week 4 ablation table to: /content/drive/MyDrive/cdss_data/results/member2_week4_ablation_summary.csv
